
두 파일의 가장 큰 차이는 **대리 모델의 알고리즘(XGBoost vs GPR)**과 **최적화 방식(Single-Objective GA vs Multi-Objective NSGA-II)**에 있습니다.

---

# 📘 반도체 패키징 대리 모델 및 최적화 기법 비교 해설서 (v1 vs v2)

이 문서는 기존의 XGBoost 기반 파이프라인(v1)과 새롭게 도입된 가우시안 프로세스 및 다목적 최적화 파이프라인(v2)의 차이점을 단계별로 분석합니다.

---

## [Step 1] 대리 모델 알고리즘의 변화 (Surrogate Modeling)

### 1. XGBoost vs 가우시안 프로세스 회귀 (GPR)

* **v1 기법:** **XGBoost (Extreme Gradient Boosting)** - 결정 트리 기반의 앙상블 학습 알고리즘을 사용했습니다.
* **v2 기법:** **GPR (Gaussian Process Regression)** - 커널 기반의 확률적 회귀 모델을 사용합니다.
* **분기점 및 차이점:**
* **데이터 효율성:** XGBoost는 데이터가 많을수록 유리하지만, GPR은 수백 개 단위의 적은 데이터셋에서도 매우 강력한 성능을 발휘합니다.
* **신뢰도 제공:** XGBoost는 예측값(한 점)만 주지만, GPR은 예측값과 함께 **'불확실성(표준편차)'**을 함께 제공합니다. 이를 통해 AI가 자신의 예측이 얼마나 정확한지를 스스로 판단할 수 있게 되었습니다.



---

## [Step 2] 데이터 샘플링 전략의 고도화 (Sampling)

### 1. 몬테카를로 vs 라틴 하이퍼큐브 샘플링 (LHS)

* **v1 기법:** **몬테카를로 샘플링 (Random Sampling)** - 단순히 무작위로 10만 개의 조합을 뽑았습니다.
* **v2 기법:** **Latin Hypercube Sampling (LHS)** - 설계 공간을 격자 형태로 나누어 중복 없이 골고루 샘플을 추출합니다.
* **분기점 및 차이점:**
* **공간 탐색 능력:** 무작위 샘플링은 특정 영역에 뭉칠 수 있지만, LHS는 6차원 설계 공간(P1~P6) 전체를 **빈틈없이 촘촘하게 훑어냅니다.** 이는 더 적은 샘플링으로도 더 정확한 최적점을 찾는 기반이 됩니다.



---

## [Step 3 ~ 5] 최적화 알고리즘의 진화 (Optimization)

### 1. 유전 알고리즘 (GA) vs NSGA-II

* **v1 기법:** **기본 유전 알고리즘 (Single-Objective GA)** - 여러 지표를 하나로 합친 점수(Fitness)가 가장 높은 단 하나의 설계안을 찾습니다.
* **v2 기법:** **NSGA-II (Non-dominated Sorting Genetic Algorithm II)** - 서로 충돌하는 여러 목표를 동시에 최적화하는 다목적 알고리즘을 사용합니다.
* **분기점 및 차이점:**
* **트레이드오프 분석:** v1은 '휨'과 '박리 응력'을 적당히 섞어서 최적화했지만, v2는 **"휨을 최소화하면서 동시에 박리 응력도 최소화하는"** 최적의 균형점들(Pareto Front)을 찾아냅니다.
* **다양한 선택지:** v1은 정답 하나만 알려주지만, v2는 '휨에 특화된 안', '안정성에 특화된 안' 등 여러 개의 상위 설계안을 사용자에게 제안합니다.



---

## [Step 6+] 결과 도출 및 의사결정 방식

### 1. 단일 확정안 vs 니 포인트 (Knee Point) 선정

* **v1 방식:** 알고리즘이 내놓은 최상위 개체 하나를 최종안으로 선택했습니다.
* **v2 방식:** 도출된 파레토 프런트(Pareto Front) 상에서 가장 효율적인 균형점인 **'니 포인트(Knee Point)'**와 강건성(Robustness) 지표를 결합하여 최종안을 선정합니다.
* **분기점 및 차이점:**
* **현실성:** v2는 GPR이 제공하는 '예측 불확실성'이 낮은 구간을 우선적으로 선택함으로써, AI가 잘 모르는 영역에서 나온 위험한 최적안을 걸러내고 **실제 공정에서 성공 확률이 높은 안**을 선택하게 됩니다.



---

## 💡 종합 비교 요약

| 구분 | v1 (Step 1 XGBoost...) | v2 (Flipchip surrogate v2...) |
| --- | --- | --- |
| **핵심 알고리즘** | XGBoost (결정 트리 앙상블) | GPR (가우시안 프로세스 회귀) |
| **샘플링 기법** | 무작위 몬테카를로 샘플링 | 라틴 하이퍼큐브 샘플링 (LHS) |
| **최적화 목표** | 단일 목표 (가중치 합산) | **다목적 최적화 (Multi-Objective)** |
| **최적화 도구** | Simple GA | **NSGA-II** |
| **강점** | 대량 데이터 처리 및 빠른 속도 | **적은 데이터로 고정밀 예측 & 신뢰도 분석** |

**결론적으로 v2는 v1에 비해 데이터 탐색 효율이 높고, 물리적으로 상충하는 목표들 사이의 정교한 균형을 잡는 능력이 대폭 강화된 버전입니다.**

두 파일의 **[Step 6] 결과값 수치**를 비교하면, AI 모델이 제안한 '최적 설계안'의 방향성과 그로 인한 성능 개선 폭에서 뚜렷한 차이가 나타납니다.

v1(**XGBoost**)은 극단적인 수치 변화를 통해 특정 지표를 억제하는 데 집중한 반면, v2(**NSGA-II + GPR**)는 여러 지표 사이의 균형을 맞춘 '현실적인 최적점'을 찾아냈습니다. 상세한 수치 차이 해설은 다음과 같습니다.

---

### 1. 설계 변수(P1~P6) 최적화 수치 비교

| 설계 변수 | v1 (XGBoost + GA) 선정 수치 | v2 (GPR + NSGA-II) 선정 수치 | 비교 및 해석 |
| --- | --- | --- | --- |
| **P1 (Top Encap)** | **0.0653 mm (최소화)** | **0.8000 mm (범위 하한)** | v1은 모델의 예측 범위를 넘어설 정도로 P1을 극단적으로 얇게 줄여 휨을 억제하려 했습니다. v2는 물리적 신뢰도가 높은 범위 내의 최저점을 선택했습니다. |
| **P4 (Die Attach)** | **0.8 ~ 0.9 mm (보강)** | **0.2900 mm (범위 상한)** | v1은 하부 지지력을 극대화하기 위해 P4를 두껍게 가져갔으나, v2는 실제 공정 범위를 준수하며 최적의 두께를 결정했습니다. |
| **P2, P6** | 블록화 (두께 증가) | 균형 잡힌 수치 도출 | v1은 구조적 안정성을 위해 보강하는 방향을 택했고, v2는 다목적 최적화를 통해 응력 분산에 최적화된 수치를 도출했습니다. |

---

### 2. 주요 성능 지표(Y 변수) 결과 비교

#### ① 휨 제어 성능 (WarpMax)

* **v1 결과:** 수치상으로는 휨이 매우 낮게 예측되었으나, 이는 P1을 극단적으로 얇게 설정한 '공격적인' 결과입니다.
* **v2 결과:** GPR 모델의 **'Knee Point(무릎 점)'**를 선택하여, 휨을 충분히 억제하면서도 다른 응력 지표가 튀지 않는 안정적인 수치를 보여줍니다.
* **해설:** v1은 '휨 최소화'라는 한 가지 목표에 매몰될 위험이 있었으나, v2는 다목적 최적화를 통해 현실적으로 납득 가능한 수준의 휨 제어치를 제시합니다.

#### ② 박리 위험성 (T_Tip_Peel)

* **v1 결과:** 특정 가중치 설정에 따라 박리 응력이 낮아졌으나, 변수 변화폭이 커서 실제 제조 시 안정성은 확인이 어렵습니다.
* **v2 결과:** **강건성(Robustness)** 지표가 반영되어, 공정 오차가 발생하더라도 박리 응력이 급격히 높아지지 않는 구간의 수치를 최종 낙점했습니다.
* **해설:** v2에서 도출된 수치는 단순 성능 최적화를 넘어, 실제 생산 시 불량률까지 고려된 **'안전한 최적치'**라는 점에서 v1과 수치적 신뢰도 차이가 발생합니다.

---

### 3. 결론: 왜 이런 차이가 생겼는가?

1. **예측의 보수성 vs 공격성:** XGBoost(v1)는 데이터 패턴에 따라 공격적인 수치를 내놓는 경향이 있는 반면, 가우시안 프로세스(v2)는 불확실성을 계산하므로 **데이터가 확실한 영역(안전한 수치)** 내에서 답을 찾으려 합니다.
2. **단일 정답 vs 균형 잡힌 집합:** v1은 사용자가 정한 가중치에 따른 **'단 하나의 수치'**를 목표로 진화했지만, v2는 여러 목표가 충돌하는 지점을 분석하여 **'가장 효율적인 절충점'**의 수치를 선택했기 때문입니다.

**요약하자면, v1의 결과값은 "이론상 최대 성능"에 가깝고, v2의 결과값은 "실제 양산 가능한 최선의 균형점"을 의미하는 수치들입니다.**